## Preparation

In [3]:
# import pandas as pd
import pyspark
from pyspark.sql import SparkSession

## Question 1: Install Spark and PySpark

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/16 09:59:24 WARN Utils: Your hostname, codespaces-e8ab3a, resolves to a loopback address: 127.0.0.1; using 10.0.7.124 instead (on interface eth0)
26/03/16 09:59:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 09:59:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [56]:
spark.version

'4.1.1'

In [5]:
df_yellow = spark.read.parquet('data/yellow_tripdata_2025-11.parquet')

In [6]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Question 2: Repartition

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

Answer: 25 MB

In [10]:
df_yellow \
        .repartition(4) \
        .write.parquet('data/raw/', mode = 'overwrite')

## Question 3: Count records

How many taxi trips were there on the 15th of November?

Answer: 162604

In [14]:
df_yellow.registerTempTable('df_yelllow_data')

/workspaces/data-engineering-zoomcamp/homework/ch6-batch-spark/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [18]:
spark.sql("""
SELECT
    DATE(tpep_pickup_datetime) AS date_trip,
    count(1)
FROM
    df_yelllow_data
WHERE 
    DATE(tpep_pickup_datetime) = '2025-11-15'
GROUP BY
    1
""").show()

+----------+--------+
| date_trip|count(1)|
+----------+--------+
|2025-11-15|  162604|
+----------+--------+



## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

Answer: 90.6 hours

In [28]:
spark.sql("""
SELECT
ROUND(TIMESTAMPDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) / 60, 2) as diff_hr
FROM df_yelllow_data
ORDER BY 1 DESC
LIMIT 10
""").show()

[Stage 19:>                                                         (0 + 2) / 2]

+-------+
|diff_hr|
+-------+
|  90.63|
|  76.93|
|   76.2|
|  69.28|
|  67.07|
|  63.37|
|  56.37|
|  48.13|
|  47.47|
|  45.43|
+-------+



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

Answer: 4040

## Question 6: Least frequent pickup location zone

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

Answer: Governor's Island/Ellis Island/Liberty Island

In [55]:
from pyspark.sql import functions as F

In [33]:
df_zone = spark.read.csv("data/taxi_zone_lookup.csv", header = True)

In [47]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [34]:
df_zone.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [54]:
df_yellow \
    .join(df_zone, df_yellow["PULocationID"] == df_zone["LocationID"], "left") \
    .groupBy("PULocationID","Zone") \
    .count() \
    .withColumnRenamed("count", "freq") \
    .orderBy("freq", ascending=True) \
    .show()

[Stage 31:>                                                         (0 + 2) / 2]

+------------+--------------------+----+
|PULocationID|                Zone|freq|
+------------+--------------------+----+
|         105|Governor's Island...|   1|
|          84|Eltingville/Annad...|   1|
|           5|       Arden Heights|   1|
|         187|       Port Richmond|   3|
|         204|   Rossville/Woodrow|   4|
|         199|       Rikers Island|   4|
|         111| Green-Wood Cemetery|   4|
|         109|         Great Kills|   4|
|           2|         Jamaica Bay|   5|
|         251|         Westerleigh|  12|
|         176|             Oakwood|  14|
|         172|New Dorp/Midland ...|  14|
|          59|        Crotona Park|  14|
|         245|       West Brighton|  14|
|         253|       Willets Point|  15|
|          27|Breezy Point/Fort...|  16|
|         206|Saint George/New ...|  17|
|          30|       Broad Channel|  18|
|         156|     Mariners Harbor|  21|
|         118|Heartland Village...|  22|
+------------+--------------------+----+
only showing top